# Chapter 2: From a candidate to a confirmed finding

**The question.** The Chapter 2 notebook in `regional_case` follows one test through the search loop using saved records. This notebook shows the outcome when the whole study was run again from scratch, with new search seeds, a live language-model proposal and a new confirmation sample. Did the loop still produce weak regions that hold up on new customers?

**What was run live.** Run 20260923T220909Z repeated the whole study on two machines. `spark-fe12` ran the numerical search with three new seeds (74101 to 74103), one live structural proposal from a language model and 200,000 new confirmation customers. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The simulated population and the frozen money-laundering model are the same as in the `regional_case` notebooks.

**Result.** On the new customers every searched region is a weaker spot than the single-clustering reference. The search that used the language model's proposal was the best at one seed of three.

**Terms.** *Share*: the fraction of customers in a region. *Contrast*: how much worse the model does inside the region than outside it (the difference in average Brier loss); higher means a weaker spot. *Arm*: one way of running the search.

The seeds, the proposal and the confirmation sample are new, so do not mix these numbers with those in the `regional_case` notebooks. This notebook reads recorded results; rerunning it makes no language-model call.

## Setup and source checks

Finds the saved results of this run and checks their SHA-256 fingerprint, so every value below comes from the unchanged run.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## The frozen finalists on new customers

The cell prints the two search goals and the rule that picks each search's final answer, counts the frozen regions and shows each one measured on the 200,000 new customers. The regions were frozen before those customers were generated.

In [2]:
protocol=read('regional/protocol.json')
print(protocol['objectives'])
print(protocol['primary_policy'])
print('Frozen candidates:',len(read('regional/frozen_finalists.json')['finalists']))
display(pd.DataFrame(results['confirmation']).T[['share','brier','contrast']])

['largest eligible region Brier contrast versus pooled rest', 'selected region population share']
maximum discovery contrast on Pareto frontier with share >= 0.10; tie larger share then earlier ID
Frozen candidates: 13


,share,brier,contrast
twinning_74101,0.10317,0.074524,0.05968
random_74101,0.10317,0.074524,0.05968
continuation_74101,0.10317,0.074524,0.05968
reflection_74101,0.106785,0.085772,0.072514
twinning_74102,0.099985,0.074805,0.059781
random_74102,0.099985,0.074805,0.059781
continuation_74102,0.099985,0.074805,0.059781
reflection_74102,0.13352,0.071811,0.058639
twinning_74103,0.111095,0.077796,0.063892
random_74103,0.11379,0.07194,0.057479


**Read the output.** **In one line: every searched region is a weaker spot than the single-clustering reference, and the language-model arm was the best at one seed of three.**

**Printed lines.** The two search goals are the contrast of the worst eligible region and that region's share. The final answer is the region with the largest contrast on the frontier among those with at least 10% share.

**Columns.** Each row is one frozen region, named arm and seed. `twinning`, `random` and `continuation` are the ordinary search started from twinning subsets, started from random subsets and given a second budget. `reflection` is the search after the language model's admitted proposal, a new feature equal to business-receipt share times cash intensity. `pam` is a single clustering without search. `share`: fraction of new customers in the region. `brier`: the model's average error inside it. `contrast`: inside minus outside.

**Notice.**

- At seeds 74101 and 74102 the three ordinary arms froze the same region, so their rows are identical (contrast 0.05968 and 0.059781).
- The reflection arm froze a different region at every seed, with contrast 0.072514, 0.058639 and 0.070908. It is the best at seed 74101 only; at seed 74103 continuation is slightly higher (0.07181).
- `pam` has the lowest contrast, 0.042629, and the broadest region, 20.1% of customers.
- One region covers 9.9985% of the new customers, just under 10%. The 10% rule applies when the answer is chosen on the discovery customers; the frozen region is not adjusted afterwards.